In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/26 11:14:40 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/02/26 11:14:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 11:14:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Если говорить проще, pivot() используется для того, чтобы значения внутри одного столбца стали новыми столбцами.

Синтаксис

pivot_df = original_df.groupBy("grouping_column").pivot("pivot_column").agg({"agg_column": "agg_function"})

                  
grouping_column: Столбец, используемый для группировки.

pivot_column: Столбец, отдельные значения которого становятся новыми столбцами.

agg_column: Столбец, для которого применяется агрегирование (например, с помощью функций sum, avg и т. д.).

In [2]:
data = [("Banana",1000,"USA"), ("Carrots",1500,"USA"), ("Beans",1600,"USA"), \
      ("Orange",2000,"USA"),("Orange",2000,"USA"),("Banana",400,"China"), \
      ("Carrots",1200,"China"),("Beans",1500,"China"),("Orange",4000,"China"), \
      ("Banana",2000,"Canada"),("Carrots",2000,"Canada"),("Beans",2000,"Mexico")]

columns= ["Product","Amount","Country"]
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show(truncate=False)

root
 |-- Product: string (nullable = true)
 |-- Amount: long (nullable = true)
 |-- Country: string (nullable = true)



+-------+------+-------+
|Product|Amount|Country|
+-------+------+-------+
|Banana |1000  |USA    |
|Carrots|1500  |USA    |
|Beans  |1600  |USA    |
|Orange |2000  |USA    |
|Orange |2000  |USA    |
|Banana |400   |China  |
|Carrots|1200  |China  |
|Beans  |1500  |China  |
|Orange |4000  |China  |
|Banana |2000  |Canada |
|Carrots|2000  |Canada |
|Beans  |2000  |Mexico |
+-------+------+-------+



PySpark SQL предоставляет функцию pivot() для преобразования данных из одного столбца в несколько столбцов. 

Это агрегация, при которой значения одного из столбцов группировки переносятся в отдельные столбцы с разными данными. 

Чтобы получить общее количество "Amount" экспортированного в каждую страну для каждого продукта, выполним группировку по продукту, pivot по стране и сумму Amount.

In [3]:
# Applying pivot()
pivotDF = df.groupBy("Product").pivot("Country").sum("Amount")
pivotDF.printSchema()
pivotDF.show(truncate=False)

root
 |-- Product: string (nullable = true)
 |-- Canada: long (nullable = true)
 |-- China: long (nullable = true)
 |-- Mexico: long (nullable = true)
 |-- USA: long (nullable = true)

+-------+------+-----+------+----+
|Product|Canada|China|Mexico|USA |
+-------+------+-----+------+----+
|Orange |NULL  |4000 |NULL  |4000|
|Beans  |NULL  |1500 |2000  |1600|
|Banana |2000  |400  |NULL  |1000|
|Carrots|2000  |1200 |NULL  |1500|
+-------+------+-----+------+----+



В версии 2.0 и выше производительность Pivot была улучшена, однако, если вы используете более раннюю версию, обратите внимание, что pivot - очень дорогая операция. 

Поэтому рекомендуется предоставлять данные столбцов (если они известны) в качестве аргумента функции, как показано ниже.

In [4]:
countries = ["USA","China","Canada","Mexico"]
pivotDF = df.groupBy("Product").pivot("Country", countries).sum("Amount")
pivotDF.show(truncate=False)

+-------+----+-----+------+------+
|Product|USA |China|Canada|Mexico|
+-------+----+-----+------+------+
|Orange |4000|4000 |NULL  |NULL  |
|Beans  |1600|1500 |NULL  |2000  |
|Banana |1000|400  |2000  |NULL  |
|Carrots|1500|1200 |2000  |NULL  |
+-------+----+-----+------+------+



Unpivot
---

Unpivot - это обратная операция, которую можно выполнить, превратив значения столбцов в значения строк. 

В PySpark SQL нет функции unpivot, поэтому мы будем использовать функцию stack(). Приведенный ниже код преобразует страны столбцов в строки.

In [6]:
# Applying unpivot()

unpivotExpr = "stack(3, 'Canada', Canada, 'China', China, 'Mexico', Mexico) as (Country,Total)"
unPivotDF = pivotDF.select("Product", F.expr(unpivotExpr)) \
    .where("Total is not null")

unPivotDF.show(truncate=False)

+-------+-------+-----+
|Product|Country|Total|
+-------+-------+-----+
|Orange |China  |4000 |
|Beans  |China  |1500 |
|Beans  |Mexico |2000 |
|Banana |Canada |2000 |
|Banana |China  |400  |
|Carrots|Canada |2000 |
|Carrots|China  |1200 |
+-------+-------+-----+



In [7]:
spark.stop()